# Twitter / X Scraper Toolkit (Apify)

This notebook wraps two Apify actors:

1. **`apidojo/tweet-scraper`** — Tweet Scraper V2. Best for large-scale search/profile/list scraping. **Hard minimum of 50 tweets per query** (pay-per-1000-tweets pricing) — if a query matches fewer than 50, it returns nothing at all rather than a partial result.
2. **`apidojo/twitter-scraper-lite`** — Twitter Scraper Unlimited. No minimum (event-based pricing). Best for conversation/reply threads, single tweets, and any **low-volume keyword search** that might fall under Actor 1's 50-tweet floor.

**Workflow:**
- Cell 1: install dependencies
- Cell 2: enter your Apify API token (hidden input, not stored in the notebook file)
- §3: Actor 1 — large-scale search/post scraping (≥50 expected matches)
- §4: Actor 2 — replies to a specific post (4a), low-volume keyword search (4b), single tweet/article fetch
- Final cell: save all collected results to disk with `json.dump`


## 1. Install dependencies

In [9]:
%pip install apify-client --quiet

Note: you may need to restart the kernel to use updated packages.


## 2. Enter your Apify API token

Uses `getpass` so the token isn't echoed to the notebook output or saved in the `.ipynb` file.
Get your token from the Apify Console → Settings → Integrations.

In [13]:
import getpass
from apify_client import ApifyClient

APIFY_API_TOKEN = getpass.getpass("Enter your Apify API token: ")
client = ApifyClient(APIFY_API_TOKEN)

print("Client initialized." if APIFY_API_TOKEN else "No token entered!")

Client initialized.


## 3. Actor 1 — Tweet Scraper V2 (`apidojo/tweet-scraper`)

Use this for **searching/scraping posts** — keyword search, a profile's tweets, hashtags, etc.

Notes from the actor docs:
- Minimum 50 tweets per query (don't use this actor for single tweets or reply threads — use Actor 2 for that).
- `searchTerms` accepts standard [Twitter advanced search syntax](https://github.com/igorbrigadir/twitter-advanced-search), e.g. `"from:NASA filter:media"`, `"#AI lang:en"`, `"cryptocurrency filter:verified"`.
- Leave fields at their defaults (`None` / `[]`) if you don't want to use them.

In [27]:
TWEET_SCRAPER_ACTOR = "apidojo/tweet-scraper"

def run_tweet_search(
    search_terms=None,       # list[str], e.g. ["from:NASA", "artificial intelligence"]
    start_urls=None,         # list[str], profile/search/list URLs
    twitter_handles=None,    # list[str], e.g. ["elonmusk", "NASA"]
    max_items=100,           # int or None for unlimited
    sort="Latest",           # "Top" | "Latest" | "Latest + Top"
    tweet_language=None,     # ISO 639-1 code, e.g. "en"
    only_verified_users=False,
    only_image=False,
    only_video=False,
    minimum_retweets=None,
    minimum_favorites=None,
    start_date=None,         # only applies to search_terms, e.g. "2024-01-01"
    end_date=None,
):
    """Run the Tweet Scraper V2 actor and return the list of result items."""
    run_input = {}
    if search_terms:
        run_input["searchTerms"] = search_terms
    if start_urls:
        run_input["startUrls"] = start_urls
    if twitter_handles:
        run_input["twitterHandles"] = twitter_handles
    if max_items is not None:
        run_input["maxItems"] = max_items
    if sort:
        run_input["sort"] = sort
    if tweet_language:
        run_input["tweetLanguage"] = tweet_language
    if only_verified_users:
        run_input["onlyVerifiedUsers"] = True
    if only_image:
        run_input["onlyImage"] = True
    if only_video:
        run_input["onlyVideo"] = True
    if minimum_retweets is not None:
        run_input["minimumRetweets"] = minimum_retweets
    if minimum_favorites is not None:
        run_input["minimumFavorites"] = minimum_favorites
    if start_date:
        run_input["start"] = start_date
    if end_date:
        run_input["end"] = end_date

    print("Running Tweet Scraper V2 with input:", run_input)
    run = client.actor(TWEET_SCRAPER_ACTOR).call(run_input=run_input)
    print(f"Run finished. Dataset: https://console.apify.com/storage/datasets/{run['defaultDatasetId']}")

    items = list(client.dataset(run["defaultDatasetId"]).iterate_items())
    print(f"Fetched {len(items)} items.")
    return items

### 3a. Run a search — edit the parameters below and execute

In [29]:
# --- Edit these inputs ---
search_results = run_tweet_search(
    search_terms=["permanent daylight savings"],   # e.g. ["from:NASA since:2024-01-01"], or ["artificial intelligence"]
    start_urls=None,              # e.g. ["https://x.com/NASA"]
    twitter_handles=None,         # e.g. ["NASA", "elonmusk"]
    max_items=100,
    sort="Latest + Top",
    tweet_language=None,          # e.g. "en"
    only_verified_users=False,
    only_image=False,
    only_video=False,
    minimum_retweets=None,
    minimum_favorites=None,
    start_date=None,
    end_date=None,
)

search_results[:2]  # quick peek at the first couple of results

Running Tweet Scraper V2 with input: {'searchTerms': ['permanent daylight savings'], 'maxItems': 100, 'sort': 'Latest + Top'}


[apify.tweet-scraper runId:iBuGCi1WmglOZM9Om] -> Status: RUNNING, Message: 
[apify.tweet-scraper runId:iBuGCi1WmglOZM9Om] -> 2026-07-19T23:17:01.526Z ACTOR: Pulling container image of build xKS2sJd2sqzwjsXgE from registry.
[apify.tweet-scraper runId:iBuGCi1WmglOZM9Om] -> 2026-07-19T23:17:01.528Z ACTOR: Creating container.
[apify.tweet-scraper runId:iBuGCi1WmglOZM9Om] -> 2026-07-19T23:17:01.593Z ACTOR: Starting container.
[apify.tweet-scraper runId:iBuGCi1WmglOZM9Om] -> 2026-07-19T23:17:01.595Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.tweet-scraper runId:iBuGCi1WmglOZM9Om] -> 2026-07-19T23:17:03.209Z INFO  System info {"apifyVersion":"3.4.2","apifyClientVersion":"2.12.4","crawleeVersion":"3.13.5","osType":"Linux","nodeVersion":"v22.23.1"}
[apify.tweet-scraper runId:iBuGCi1WmglOZM9Om] -> 2026-07-19T23:17:03.341Z INFO  PHASE -- STARTING ACTOR.
[apify.tweet-scraper runId:iBuGCi1WmglOZM9Om] -> 2026-07-19T23:17:03.845Z INFO  PHASE -- SETTING UP CRAWLER.
[apify.tweet-scraper runId:i

TypeError: 'Run' object is not subscriptable

## 4. Actor 2 — Twitter Scraper Unlimited (`apidojo/twitter-scraper-lite`)

No minimum result count (event-based pricing). Three uses:

- **4a. Replies to a specific post** (conversation thread) — `run_conversation_replies`. Find the conversation ID from the tweet's URL: `https://x.com/USER/status/<CONVERSATION_ID>`. Uses the `conversation_id:` search operator, which can be combined with other filters (e.g. `#AI`, `min_faves:100`).
- **4b. General keyword/profile search, low-volume-safe** — `run_keyword_search_unlimited`. Same query surface as Actor 1's `run_tweet_search` (§3), but use this one instead whenever the query might return fewer than 50 results — Actor 1 has a **hard 50-tweet-per-query minimum** and returns nothing at all if the query matches fewer, rather than partially delivering. Narrow phrases and off-peak time windows commonly fall under 50.
- Single tweet/article fetch by URL — `run_single_tweet_fetch(tweet_url)`.

In [11]:
TWITTER_LITE_ACTOR = "apidojo/twitter-scraper-lite"

def run_conversation_replies(
    conversation_id,
    extra_query=None,        # optional extra search syntax appended to the conversation_id query, e.g. "#AI" or "min_faves:10"
    max_items=100,           # int or None for unlimited
    sort="Latest",           # "Top" | "Latest" | "Latest + Top"
    tweet_language=None,
):
    """Fetch replies to a specific tweet/post via its conversation ID."""
    query = f"conversation_id:{conversation_id}"
    if extra_query:
        query += f" {extra_query}"

    run_input = {
        "searchTerms": [query],
        "sort": sort,
    }
    if max_items is not None:
        run_input["maxItems"] = max_items
    if tweet_language:
        run_input["tweetLanguage"] = tweet_language

    print("Running Twitter Scraper Unlimited with input:", run_input)
    run = client.actor(TWITTER_LITE_ACTOR).call(run_input=run_input)
    print(f"Run finished. Dataset: https://console.apify.com/storage/datasets/{run['defaultDatasetId']}")

    items = list(client.dataset(run["defaultDatasetId"]).iterate_items())
    print(f"Fetched {len(items)} reply items.")
    return items


def run_single_tweet_fetch(tweet_url):
    """Fetch a single tweet/article by URL (not supported on the other actor)."""
    run_input = {"startUrls": [tweet_url]}
    print("Running Twitter Scraper Unlimited (single fetch) with input:", run_input)
    run = client.actor(TWITTER_LITE_ACTOR).call(run_input=run_input)
    items = list(client.dataset(run["defaultDatasetId"]).iterate_items())
    print(f"Fetched {len(items)} item(s).")
    return items


def run_keyword_search_unlimited(
    search_terms=None,       # list[str], e.g. ["daylight saving time"]
    start_urls=None,         # list[str], profile/search/list URLs
    twitter_handles=None,    # list[str], e.g. ["elonmusk", "NASA"]
    max_items=100,           # int or None for unlimited
    sort="Latest",           # "Top" | "Latest" | "Latest + Top"
    tweet_language=None,     # ISO 639-1 code, e.g. "en"
    only_verified_users=False,
    minimum_retweets=None,
    minimum_favorites=None,
    minimum_replies=None,
    start_date=None,         # only applies to search_terms, e.g. "2024-01-01"
    end_date=None,
):
    """General keyword/profile search via Twitter Scraper Unlimited — same query
    surface as run_tweet_search, but no 50-tweet-per-query minimum. Use this
    instead of run_tweet_search whenever a query might return fewer than 50
    results (e.g. a narrow phrase, an off-peak time window): Tweet Scraper V2
    silently returns zero items rather than partially delivering below its
    minimum, which is a confusing way to lose a low-volume result set."""
    run_input = {}
    if search_terms:
        run_input["searchTerms"] = search_terms
    if start_urls:
        run_input["startUrls"] = start_urls
    if twitter_handles:
        run_input["twitterHandles"] = twitter_handles
    if max_items is not None:
        run_input["maxItems"] = max_items
    if sort:
        run_input["sort"] = sort
    if tweet_language:
        run_input["tweetLanguage"] = tweet_language
    if only_verified_users:
        run_input["onlyVerifiedUsers"] = True
    if minimum_retweets is not None:
        run_input["minimumRetweets"] = minimum_retweets
    if minimum_favorites is not None:
        run_input["minimumFavorites"] = minimum_favorites
    if minimum_replies is not None:
        run_input["minimumReplies"] = minimum_replies
    if start_date:
        run_input["start"] = start_date
    if end_date:
        run_input["end"] = end_date

    print("Running Twitter Scraper Unlimited (keyword search) with input:", run_input)
    run = client.actor(TWITTER_LITE_ACTOR).call(run_input=run_input)
    print(f"Run finished. Dataset: https://console.apify.com/storage/datasets/{run['defaultDatasetId']}")

    items = list(client.dataset(run["defaultDatasetId"]).iterate_items())
    print(f"Fetched {len(items)} items.")
    return items

### 4a. Fetch replies to a specific post — edit the parameters below and execute

In [26]:
# --- Edit these inputs ---
CONVERSATION_ID = "2077161715364724803"  # the numeric ID from the tweet's URL

reply_results = run_conversation_replies(
    conversation_id=CONVERSATION_ID,
    extra_query=None,     # e.g. "#AI" or "min_faves:10"
    max_items=500,
    sort="Latest + Top",
    tweet_language=None,
)

reply_results[:2]  # quick peek at the first couple of replies

Running Twitter Scraper Unlimited with input: {'searchTerms': ['conversation_id:2077161715364724803'], 'sort': 'Latest + Top', 'maxItems': 500}


[apify.twitter-scraper-lite runId:aydDJeVefQYapSMYe] -> Status: RUNNING, Message: 
[apify.twitter-scraper-lite runId:aydDJeVefQYapSMYe] -> 2026-07-19T23:10:02.917Z ACTOR: Pulling container image of build DjhB2X006cpNZMoxn from registry.
[apify.twitter-scraper-lite runId:aydDJeVefQYapSMYe] -> 2026-07-19T23:10:02.919Z ACTOR: Creating container.
[apify.twitter-scraper-lite runId:aydDJeVefQYapSMYe] -> 2026-07-19T23:10:02.955Z ACTOR: Starting container.
[apify.twitter-scraper-lite runId:aydDJeVefQYapSMYe] -> 2026-07-19T23:10:02.956Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.twitter-scraper-lite runId:aydDJeVefQYapSMYe] -> 2026-07-19T23:10:04.258Z INFO  System info {"apifyVersion":"3.4.2","apifyClientVersion":"2.12.4","crawleeVersion":"3.13.5","osType":"Linux","nodeVersion":"v22.23.1"}
[apify.twitter-scraper-lite runId:aydDJeVefQYapSMYe] -> 2026-07-19T23:10:04.367Z INFO  PHASE -- STARTING ACTOR.
[apify.twitter-scraper-lite runId:aydDJeVefQYapSMYe] -> 2026-07-19T23:10:04.651Z INFO  [

TypeError: 'Run' object is not subscriptable

### 4b. General keyword search, low-volume-safe — edit the parameters below and execute

In [ ]:
# --- Edit these inputs ---
keyword_results = run_keyword_search_unlimited(
    search_terms=["daylight saving time"],   # narrow/off-peak queries: use this fn, not run_tweet_search (§3)
    start_urls=None,
    twitter_handles=None,
    max_items=1000,
    sort="Latest",
    tweet_language="en",
    only_verified_users=False,
    minimum_retweets=None,
    minimum_favorites=None,
    minimum_replies=None,
    start_date=None,          # e.g. "2025-11-01" to target the last DST transition instead of "latest"
    end_date=None,
)

keyword_results[:2]  # quick peek at the first couple of results

## 5. Save results to disk

Writes both result sets to timestamped JSON files in the current working directory via `json.dump`.

In [ ]:
import json
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

output_files = {}

if 'search_results' in globals() and search_results:
    search_filename = f"tweet_search_results_{timestamp}.json"
    with open(search_filename, "w", encoding="utf-8") as f:
        json.dump(search_results, f, ensure_ascii=False, indent=2)
    output_files["search"] = search_filename
    print(f"Saved {len(search_results)} search results to {search_filename}")

if 'keyword_results' in globals() and keyword_results:
    keyword_filename = f"tweet_keyword_search_results_{timestamp}.json"
    with open(keyword_filename, "w", encoding="utf-8") as f:
        json.dump(keyword_results, f, ensure_ascii=False, indent=2)
    output_files["keyword_search"] = keyword_filename
    print(f"Saved {len(keyword_results)} keyword search results to {keyword_filename}")

if 'reply_results' in globals() and reply_results:
    replies_filename = f"conversation_replies_{timestamp}.json"
    with open(replies_filename, "w", encoding="utf-8") as f:
        json.dump(reply_results, f, ensure_ascii=False, indent=2)
    output_files["replies"] = replies_filename
    print(f"Saved {len(reply_results)} reply results to {replies_filename}")

print("\nFiles written:", output_files)